# Test rapide de BioMistral avec vLLM (mode hors ligne)

Notebook d'exploration : vérifier que le modèle se charge et répond. Pour les expériences, utiliser `runner.ipynb` (mode serveur + LangChain).

**Ne pas lancer ce notebook dans la même session que le serveur vLLM de `runner.ipynb`** : les deux chargent le modèle sur le GPU et le T4 n'a pas la mémoire pour deux copies.

Changements par rapport à la version initiale :
- `max_model_len` passé de 1024 à 2048 (maximum de BioMistral) ;
- seed fixée et température 0 (sorties reproductibles) ;
- `llm.chat()` au lieu d'un `[INST]` écrit à la main : le gabarit de chat du tokenizer est appliqué ;
- paramètres regroupés dans une cellule de configuration.

In [ ]:
!pip install -q vllm==0.33.0

In [ ]:
# Jeton Hugging Face depuis les Secrets Colab (icône clé) : téléchargements plus rapides
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
# ---- Configuration (aucun paramètre en dur plus bas) ----
CONFIG: dict = {
    "model": "BioMistral/BioMistral-7B-AWQ-QGS128-W4-GEMM",
    "quantization": "awq",
    "max_model_len": 2048,
    "enforce_eager": True,   # économise la mémoire sur T4 ; essayer False pour gagner en vitesse
    "seed": 42,
    "temperature": 0.0,
    "max_tokens": 200,
}

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(
    model=CONFIG["model"],
    quantization=CONFIG["quantization"],
    max_model_len=CONFIG["max_model_len"],
    enforce_eager=CONFIG["enforce_eager"],
    seed=CONFIG["seed"],
)
params = SamplingParams(
    temperature=CONFIG["temperature"],
    max_tokens=CONFIG["max_tokens"],
    seed=CONFIG["seed"],
)
print("Modèle chargé.")

In [ ]:
import time

def ask(question: str) -> str:
    """Pose une question au modèle via le gabarit de chat (pas de message system)."""
    t0 = time.perf_counter()
    out = llm.chat([{"role": "user", "content": question}], params)[0]
    n_in, n_out = len(out.prompt_token_ids), len(out.outputs[0].token_ids)
    dt = time.perf_counter() - t0
    print(f"[{n_in} tokens en entrée, {n_out} en sortie, {dt:.1f} s, {n_out / dt:.1f} tok/s]")
    return out.outputs[0].text

print(ask("Que faire pour être en bonne santé ?"))

### Petit test plus proche de la tâche

Un extrait clinique fictif : vérifier à l'œil si le modèle invente des informations.

In [ ]:
EXTRAIT = (
    "Patient de 67 ans, hypertendu, admis par les urgences pour douleur thoracique. "
    "Troponine à 1250 ng/L. Coronarographie : sténose de l'IVA traitée par angioplastie avec stent. "
    "Sortie à J5 sous aspirine 75 mg et bisoprolol 2,5 mg."
)
print(ask(f"Résume ce compte rendu en trois phrases, sans ajouter d'information :\n\n{EXTRAIT}"))